In [8]:
!git clone https://github.com/mkobycheva/recommendation-system.git 2>/dev/null \
  || (cd recommendation-system && git pull)

%cd recommendation-system
!git checkout transformer && git pull
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

/content/recommendation-system/recommendation-system
Branch 'transformer' set up to track remote branch 'transformer' from 'origin'.
Switched to a new branch 'transformer'
Already up to date.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import os
import numpy as np
import pandas as pd

from src.evaluation.metrics import ndcg_at_k, recall_at_k, map_at_k

# Data preparation

In [10]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [11]:
from src.data.build_matrix import add_domain_item_ids
from sklearn.preprocessing import LabelEncoder

books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test  = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test  = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df  = pd.concat([books_test, movies_test], ignore_index=True)

# 5-core item filtering
# Only train_df decides which items "count" — valid/test must never leak info
# about which items are frequent, they can only inherit train's decision.
MIN_ITEM_FREQ = 5
item_counts_train = train_df['item_id'].value_counts()
frequent_items = set(item_counts_train[item_counts_train >= MIN_ITEM_FREQ].index)

print(f"Items before filtering: {train_df['item_id'].nunique()}")
print(f"Items after filtering (freq >= {MIN_ITEM_FREQ}): {len(frequent_items)}")

rows_before = {name: len(df) for name, df in [('train', train_df), ('valid', valid_df), ('test', test_df)]}

train_df = train_df[train_df['item_id'].isin(frequent_items)].reset_index(drop=True)
valid_df = valid_df[valid_df['item_id'].isin(frequent_items)].reset_index(drop=True)
test_df  = test_df[test_df['item_id'].isin(frequent_items)].reset_index(drop=True)

for name, df, before in [('train', train_df, rows_before['train']),
                          ('valid', valid_df, rows_before['valid']),
                          ('test', test_df, rows_before['test'])]:
    print(f"{name}: {before} -> {len(df)} rows ({100*len(df)/before:.1f}% kept)")

# --- Rebuild vocab on the FILTERED data only, so NUM_ITEMS actually shrinks ---
encoder = LabelEncoder()
encoder.fit(pd.concat([train_df['item_id'], valid_df['item_id'], test_df['item_id']]).unique())

NUM_ITEMS = len(encoder.classes_) + 1
print(f"NUM_ITEMS = {NUM_ITEMS}")

for df in (train_df, valid_df, test_df):
    df['item_idx'] = encoder.transform(df['item_id']) + 1

Items before filtering: 587064
Items after filtering (freq >= 5): 190466
train: 3750813 -> 2894431 rows (77.2% kept)
valid: 254376 -> 181182 rows (71.2% kept)
test: 254376 -> 172979 rows (68.0% kept)
NUM_ITEMS = 190467


In [12]:
MAX_LEN = 50

def build_user_sequences(df):
    """user_id -> list of item_idx, sorted by timestamp"""
    df_sorted = df.sort_values(['user_id', 'timestamp'])
    return df_sorted.groupby('user_id')['item_idx'].apply(list).to_dict()

def generate_sliding_windows(user_sequences, max_len=MAX_LEN, min_len=2):
    """
    Next-item prediction на кожній позиції одразу (SASRec-style).
    input:  [0,...,0, i1,i2,i3,i4]
    target: [0,...,0, i2,i3,i4,i5]
    """
    X, Y = [], []
    for seq in user_sequences.values():
        if len(seq) < min_len:
            continue
        inp = seq[:-1]
        tgt = seq[1:]
        if len(inp) >= max_len:
            inp = inp[-max_len:]
            tgt = tgt[-max_len:]
        else:
            pad = max_len - len(inp)
            inp = [0] * pad + inp
            tgt = [0] * pad + tgt
        X.append(inp)
        Y.append(tgt)
    return np.array(X), np.array(Y)

def build_val_eval_batch(user_sequences, max_len=MAX_LEN):
    """Isolates only the true held-out validation target per user,
    mirroring build_test_eval_batch — avoids contamination from
    train-only transitions inside the combined sequence."""
    X, Y = [], []
    for seq in user_sequences.values():
        if len(seq) < 2:
            continue
        inp = seq[:-1]
        target = seq[-1]
        if len(inp) >= max_len:
            inp = inp[-max_len:]
        else:
            pad = max_len - len(inp)
            inp = [0] * pad + inp
        X.append(inp)
        Y.append(target)
    return np.array(X), np.array(Y)

train_sequences = build_user_sequences(train_df)
X_train, Y_train = generate_sliding_windows(train_sequences)

valid_combined = pd.concat([train_df, valid_df])
valid_sequences = build_user_sequences(valid_combined)
X_val, y_val = build_val_eval_batch(valid_sequences, max_len=MAX_LEN)  # renamed Y_val -> y_val (single target per user, not per-position)

print(f"Train: X={X_train.shape}, Y={Y_train.shape}")
print(f"Valid: X={X_val.shape}, y={len(y_val)}")

Train: X=(127132, 50), Y=(127132, 50)
Valid: X=(127170, 50), y=127170


In [13]:
non_pad_train = (Y_train != 0).sum()
total_train = Y_train.size
print(f"Non-padding targets: {non_pad_train} out of {total_train} ({100*non_pad_train/total_train:.1f}%)")

Non-padding targets: 2204414 out of 6356600 (34.7%)


# SASRec

In [14]:
import torch
from torch.utils.data import Dataset, DataLoader

class SASRecDataset(Dataset):
    """Wraps padded (input, target) sequences for next-item prediction."""

    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.Y = torch.tensor(Y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


BATCH_SIZE = 128
train_loader = DataLoader(SASRecDataset(X_train, Y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_val, dtype=torch.long), torch.tensor(y_val, dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=False
)  # y_val is one target per user now, not a full sequence -> use TensorDataset, same as test_loader

for x_batch, y_batch in train_loader:
    print(f"X batch shape: {x_batch.shape}, Y batch shape: {y_batch.shape}")
    break

X batch shape: torch.Size([128, 50]), Y batch shape: torch.Size([128, 50])


In [15]:
# Build a train-eval loader mirroring build_val_eval_batch: only the LAST position
# per user, so train MAP@5 is measured the same way as val MAP@5 (apples-to-apples).
# Using the full per-position X_train/Y_train for MAP@5 would be misleadingly easy,
# since many of those positions are the same transitions the model was trained on.
X_train_eval, y_train_eval = build_val_eval_batch(train_sequences, max_len=MAX_LEN)

train_eval_loader = DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_train_eval, dtype=torch.long), torch.tensor(y_train_eval, dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=False
)

print(f"Train-eval: X={X_train_eval.shape}, y={len(y_train_eval)}")

Train-eval: X=(127132, 50), y=127132


In [16]:
import torch.nn as nn

class SASRec(nn.Module):
    def __init__(self, num_items, max_len=MAX_LEN, d_model=64, n_heads=2, n_layers=2, dropout=0.2, emb_dropout=0.5):
        super().__init__()
        self.item_emb = nn.Embedding(num_items, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        self.max_len = max_len

    def forward(self, item_seq):
        batch_size, seq_len = item_seq.shape
        positions = torch.arange(seq_len, device=item_seq.device).unsqueeze(0).expand(batch_size, -1)

        x = self.item_emb(item_seq) + self.pos_emb(positions)
        x = self.emb_dropout(x)

        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(item_seq.device)
        padding_mask = (item_seq == 0)

        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=padding_mask)

        # padding-only query positions can produce NaN internally (fully-masked attention row);
        # those positions are excluded from the loss anyway, so zeroing NaNs here is safe
        # and stops them from contaminating real positions in later computations
        x = torch.nan_to_num(x, nan=0.0)

        x = self.layer_norm(x)
        return x

    def predict_logits(self, hidden_states):
        return hidden_states @ self.item_emb.weight.T

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SASRec(num_items=NUM_ITEMS, max_len=MAX_LEN, d_model=64, n_heads=2, n_layers=2, dropout=0.2).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1, factor=0.5)

print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: cuda
Model parameters: 12,293,184


In [18]:
item_domain = np.array(['pad'] + [
    'books' if c.startswith('books::') else 'movies'
    for c in encoder.classes_
])

domain_id = torch.tensor(
    [0 if d == 'pad' else (1 if d == 'books' else 2) for d in item_domain],
    device=device
)
books_item_idx = torch.tensor(np.where(item_domain == 'books')[0], device=device)
movies_item_idx = torch.tensor(np.where(item_domain == 'movies')[0], device=device)

print(f"Books items: {len(books_item_idx)}, Movies items: {len(movies_item_idx)}")

Books items: 106241, Movies items: 84225


In [19]:
# # Popularity-weighted negative sampling: negatives should be sampled proportional
# # to how often each item appears in train, not uniformly. Uniform negatives are
# # almost always obscure long-tail items — trivially easy to distinguish from the
# # true target — so the model never learns to out-rank genuinely popular competitors,
# # which is exactly why it currently fails to beat the popularity baseline.
# item_pop_counts = train_df['item_idx'].value_counts()
# full_pop_counts = np.zeros(NUM_ITEMS)
# full_pop_counts[item_pop_counts.index.values] = item_pop_counts.values
# full_pop_counts_t = torch.tensor(full_pop_counts, device=device, dtype=torch.float)

# # smoothing exponent (0.75, as in word2vec negative sampling) prevents the
# # most popular items from dominating the negative pool entirely
# POP_SMOOTHING = 0.75
# smoothed_pop = full_pop_counts_t.clamp(min=1.0) ** POP_SMOOTHING

# books_neg_weights = smoothed_pop[books_item_idx]
# movies_neg_weights = smoothed_pop[movies_item_idx]

# print(f"Books weight range: {books_neg_weights.min().item():.1f} - {books_neg_weights.max().item():.1f}")
# print(f"Movies weight range: {movies_neg_weights.min().item():.1f} - {movies_neg_weights.max().item():.1f}")

In [20]:
def sampled_logits_and_labels(model, hidden_valid, targets_valid, num_items, num_negatives=100):
    N = hidden_valid.size(0)
    device = hidden_valid.device

    # Sample negatives from the SAME domain as the target, so the model can't
    # shortcut on "is this a book or a movie" instead of learning real relevance
    target_domain_id = domain_id[targets_valid]
    is_books = target_domain_id == 1

    books_sample_idx = torch.randint(0, len(books_item_idx), (N, num_negatives), device=device)
    movies_sample_idx = torch.randint(0, len(movies_item_idx), (N, num_negatives), device=device)
    neg_items = torch.where(
        is_books.unsqueeze(1),
        books_item_idx[books_sample_idx],
        movies_item_idx[movies_sample_idx],
    )

    collision = neg_items == targets_valid.unsqueeze(1)
    if collision.any():
        n_coll = int(collision.sum().item())
        fallback_books = books_item_idx[torch.randint(0, len(books_item_idx), (n_coll,), device=device)]
        fallback_movies = movies_item_idx[torch.randint(0, len(movies_item_idx), (n_coll,), device=device)]
        is_books_flat = is_books.unsqueeze(1).expand(-1, num_negatives)[collision]
        neg_items[collision] = torch.where(is_books_flat, fallback_books, fallback_movies)

    candidates = torch.cat([targets_valid.unsqueeze(1), neg_items], dim=1)   # (N, 1+K)
    cand_emb = model.item_emb(candidates)                                    # (N, 1+K, d_model)

    logits = torch.bmm(cand_emb, hidden_valid.unsqueeze(-1)).squeeze(-1)     # (N, 1+K)
    labels = torch.zeros(N, dtype=torch.long, device=device)                 # correct item is always at index 0

    return logits, labels

In [21]:
@torch.no_grad()
def evaluate_map_at_k(model, hidden_valid, targets_valid, num_items, num_negatives=100, k=5):
    """
    Builds candidate-pool rankings and scores them with the repo's map_at_k
    (relies on average_precision_at_k under the hood).
    Each "user" here is really one (position, target) example — with a single
    relevant item — matching the interface map_at_k expects.
    """
    model.eval()
    N = hidden_valid.size(0)
    device = hidden_valid.device

    neg_items = torch.randint(1, num_items, (N, num_negatives), device=device)
    collision = neg_items == targets_valid.unsqueeze(1)
    if collision.any():
        neg_items[collision] = torch.randint(1, num_items, (int(collision.sum().item()),), device=device)

    candidates = torch.cat([targets_valid.unsqueeze(1), neg_items], dim=1)   # (N, 1+K)
    cand_emb = model.item_emb(candidates)
    logits = torch.bmm(cand_emb, hidden_valid.unsqueeze(-1)).squeeze(-1)      # (N, 1+K)

    top_k_idx = logits.topk(k, dim=1).indices                                # (N, k), indices into candidate axis
    top_k_items = torch.gather(candidates, 1, top_k_idx)                     # (N, k), actual item ids

    recommended_items_by_user = {i: top_k_items[i].tolist() for i in range(N)}
    relevant_items_by_user = {i: [targets_valid[i].item()] for i in range(N)}

    return map_at_k(recommended_items_by_user, relevant_items_by_user, k=k)

In [22]:
@torch.no_grad()
def sampled_ranking_map_at_k(model, hidden_all, targets_all, num_items, num_negatives=999, k=5, seed=42, chunk_size=2048):
    model.eval()
    device = hidden_all.device
    generator = torch.Generator(device=device).manual_seed(seed)

    recommended_items_by_user = {}
    relevant_items_by_user = {}
    global_idx = 0

    for start in range(0, hidden_all.size(0), chunk_size):
        end = start + chunk_size
        hidden_chunk = hidden_all[start:end]
        targets_chunk = targets_all[start:end]
        n = hidden_chunk.size(0)

        target_domain_id_chunk = domain_id[targets_chunk]
        is_books_chunk = target_domain_id_chunk == 1
        books_sample_idx = torch.randint(0, len(books_item_idx), (n, num_negatives), device=device, generator=generator)
        movies_sample_idx = torch.randint(0, len(movies_item_idx), (n, num_negatives), device=device, generator=generator)
        neg_items = torch.where(
            is_books_chunk.unsqueeze(1),
            books_item_idx[books_sample_idx],
            movies_item_idx[movies_sample_idx],
        )

        collision = neg_items == targets_chunk.unsqueeze(1)
        if collision.any():
            n_coll = int(collision.sum().item())
            fallback_books = books_item_idx[torch.randint(0, len(books_item_idx), (n_coll,), device=device, generator=generator)]
            fallback_movies = movies_item_idx[torch.randint(0, len(movies_item_idx), (n_coll,), device=device, generator=generator)]
            is_books_flat = is_books_chunk.unsqueeze(1).expand(-1, num_negatives)[collision]
            neg_items[collision] = torch.where(is_books_flat, fallback_books, fallback_movies)

        candidates = torch.cat([targets_chunk.unsqueeze(1), neg_items], dim=1)   # (n, 1+K)
        cand_emb = model.item_emb(candidates)
        logits = torch.bmm(cand_emb, hidden_chunk.unsqueeze(-1)).squeeze(-1)      # (n, 1+K)

        top_k_idx = logits.topk(k, dim=1).indices
        top_k_items = torch.gather(candidates, 1, top_k_idx)

        for i in range(n):
            recommended_items_by_user[global_idx] = top_k_items[i].tolist()
            relevant_items_by_user[global_idx] = [targets_chunk[i].item()]
            global_idx += 1

    return map_at_k(recommended_items_by_user, relevant_items_by_user, k=k)

## Training

In [23]:
import wandb

wandb.init(
    project="recsys-sequential",
    name="sasrec-v4-dmodel64",
    config={
        "model": "SASRec",
        "max_len": MAX_LEN,
        "d_model": 64,
        "n_heads": 2,
        "n_layers": 2,
        "dropout": 0.2,
        "batch_size": BATCH_SIZE,
        "lr": 1e-3,
        "weight_decay": 1e-4,
    }
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: recommender-ai28 (recommender-ai28-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [24]:
EPOCHS = 30
PATIENCE = 5
MIN_DELTA = 0.0
MIN_EPOCHS_BEFORE_STOPPING = 10
WARMUP_EPOCHS = 5
best_val_map = -float('inf')
patience_counter = 0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_grad_norm = 0.0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        hidden = model(x_batch)

        # Flatten and keep only non-padding positions BEFORE the expensive matmul
        hidden_flat = hidden.view(-1, hidden.size(-1))
        targets_flat = y_batch.view(-1)
        mask = targets_flat != 0
        hidden_valid = hidden_flat[mask]
        targets_valid = targets_flat[mask]

        logits, labels = sampled_logits_and_labels(model, hidden_valid, targets_valid, NUM_ITEMS, num_negatives=100)
        loss = criterion(logits, labels)
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        total_grad_norm += grad_norm.item()

        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    avg_grad_norm = total_grad_norm / len(train_loader)

    model.eval()
    val_loss = 0.0
    all_hidden = []
    all_targets = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            hidden = model(x_batch)

            # y_batch is now a single held-out target per user -> take only the LAST position's hidden state,
            # matching how build_val_eval_batch left-pads (last position = last real history item -> predicts y_batch)
            hidden_valid = hidden[:, -1, :]
            targets_valid = y_batch

            all_hidden.append(hidden_valid.cpu())
            all_targets.append(targets_valid.cpu())

            logits, labels = sampled_logits_and_labels(model, hidden_valid, targets_valid, NUM_ITEMS, num_negatives=100)
            loss = criterion(logits, labels)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    all_hidden = torch.cat(all_hidden).to(device)
    all_targets = torch.cat(all_targets).to(device)
    val_map5 = sampled_ranking_map_at_k(model, all_hidden, all_targets, NUM_ITEMS, num_negatives=999, k=5, seed=42)

    # Train MAP@5 for overfit tracking — same last-position-only protocol as val,
    # so the train/val gap is a real generalization gap, not a measurement artifact
    all_hidden_train_eval, all_targets_train_eval = [], []
    with torch.no_grad():
        for x_batch, y_batch in train_eval_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            hidden = model(x_batch)
            all_hidden_train_eval.append(hidden[:, -1, :].cpu())
            all_targets_train_eval.append(y_batch.cpu())

    all_hidden_train_eval = torch.cat(all_hidden_train_eval).to(device)
    all_targets_train_eval = torch.cat(all_targets_train_eval).to(device)
    train_map5 = sampled_ranking_map_at_k(model, all_hidden_train_eval, all_targets_train_eval, NUM_ITEMS, num_negatives=999, k=5, seed=42)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "grad_norm": avg_grad_norm,
        "learning_rate": current_lr,
        "val_map@5": val_map5,
        "train_map@5": train_map5,
        "overfit_gap_map@5": train_map5 - val_map5,
    })

    print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train MAP@5: {train_map5:.4f} | Val MAP@5: {val_map5:.4f} | Gap: {train_map5 - val_map5:.4f} | Grad Norm: {avg_grad_norm:.4f}")

    if epoch + 1 > WARMUP_EPOCHS and val_map5 > best_val_map + MIN_DELTA:
        best_val_map = val_map5
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if epoch + 1 >= MIN_EPOCHS_BEFORE_STOPPING and patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(best_state)
wandb.finish()

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


Epoch [1/30] Train Loss: 7.2283 | Val Loss: 5.9299 | Train MAP@5: 0.0023 | Val MAP@5: 0.0026 | Gap: -0.0003 | Grad Norm: 1.3632
Epoch [2/30] Train Loss: 4.5855 | Val Loss: 4.8428 | Train MAP@5: 0.0044 | Val MAP@5: 0.0047 | Gap: -0.0003 | Grad Norm: 0.1438
Epoch [3/30] Train Loss: 4.2841 | Val Loss: 4.4797 | Train MAP@5: 0.0275 | Val MAP@5: 0.0259 | Gap: 0.0015 | Grad Norm: 0.1115
Epoch [4/30] Train Loss: 4.1204 | Val Loss: 4.3778 | Train MAP@5: 0.0504 | Val MAP@5: 0.0459 | Gap: 0.0045 | Grad Norm: 0.1614
Epoch [5/30] Train Loss: 3.9187 | Val Loss: 4.3571 | Train MAP@5: 0.0540 | Val MAP@5: 0.0485 | Gap: 0.0055 | Grad Norm: 0.3348
Epoch [6/30] Train Loss: 3.7179 | Val Loss: 4.3727 | Train MAP@5: 0.0538 | Val MAP@5: 0.0481 | Gap: 0.0057 | Grad Norm: 0.5282
Epoch [7/30] Train Loss: 3.5372 | Val Loss: 4.3860 | Train MAP@5: 0.0566 | Val MAP@5: 0.0506 | Gap: 0.0059 | Grad Norm: 0.7190
Epoch [8/30] Train Loss: 3.3034 | Val Loss: 4.3910 | Train MAP@5: 0.0592 | Val MAP@5: 0.0526 | Gap: 0.0066 | 

epoch,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
grad_norm,█▁▁▁▂▃▄▅▅▆▆▇▇▇███
learning_rate,██████▄▄▃▃▂▂▁▁▁▁▁
overfit_gap_map@5,▁▁▂▄▅▅▅▆▆▆▇▇▇████
train_loss,█▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
train_map@5,▁▁▄▇▇▇▇██████████
val_loss,█▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_map@5,▁▁▄▇▇▇███████████
epoch,17
grad_norm,1.44029
learning_rate,2e-05


## Validate on test set

In [25]:
test_combined = pd.concat([train_df, valid_df, test_df])
test_sequences = build_user_sequences(test_combined)

def build_test_eval_batch(user_sequences, max_len=MAX_LEN):
    """
    For each user: history = everything except the last interaction,
    target = the last interaction (the held-out test item to predict).
    Left-padding guarantees the last position in each row is always
    the last real history item (never padding), as long as history has >=1 item.
    """
    X, Y = [], []
    for seq in user_sequences.values():
        if len(seq) < 2:
            continue
        inp = seq[:-1]
        target = seq[-1]
        if len(inp) >= max_len:
            inp = inp[-max_len:]
        else:
            pad = max_len - len(inp)
            inp = [0] * pad + inp
        X.append(inp)
        Y.append(target)
    return np.array(X), np.array(Y)

X_test, y_test = build_test_eval_batch(test_sequences)
print(f"Test: X={X_test.shape}, y={len(y_test)}")

Test: X=(127184, 50), y=127184


In [26]:
model.eval()

test_loader = DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_test, dtype=torch.long), torch.tensor(y_test, dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=False
)

all_hidden_test, all_targets_test = [], []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        hidden = model(x_batch)
        all_hidden_test.append(hidden[:, -1, :].cpu())   # last position = prediction for the held-out item
        all_targets_test.append(y_batch)

all_hidden_test = torch.cat(all_hidden_test).to(device)
all_targets_test = torch.cat(all_targets_test).to(device)

print(f"all_hidden_test: {all_hidden_test.shape}, all_targets_test: {all_targets_test.shape}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


all_hidden_test: torch.Size([127184, 64]), all_targets_test: torch.Size([127184])


In [27]:
def average_metric_over_users(metric_fn, recommended_by_user, relevant_by_user, k):
    """Applies a per-user metric function (ndcg_at_k, recall_at_k, precision_at_k)
    across a dict of users and averages the result — mirrors what map_at_k does
    internally via average_precision_at_k, since those other functions don't have
    their own dict-aware wrapper in the repo.
    """
    scores = [
        metric_fn(recommended_by_user.get(user_id, []), relevant_items, k=k)
        for user_id, relevant_items in relevant_by_user.items()
    ]
    return sum(scores) / len(scores) if scores else 0.0

In [28]:
import gc

if 'optimizer' in globals():
    del optimizer
if 'scheduler' in globals():
    del scheduler

gc.collect()
torch.cuda.empty_cache()

print(f"Free GPU memory: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

@torch.no_grad()
def full_ranking_eval(model, hidden_all, targets_all, num_items, k_list=(5, 10), chunk_size=2048):
    """Returns top-k recommended items dict + relevant items dict, ready for map_at_k/ndcg_at_k/recall_at_k."""
    max_k = max(k_list)
    all_item_emb = model.item_emb.weight
    recommended_items_by_user = {}
    relevant_items_by_user = {}
    global_idx = 0

    for start in range(0, hidden_all.size(0), chunk_size):
        end = start + chunk_size
        hidden_chunk = hidden_all[start:end]
        targets_chunk = targets_all[start:end]

        logits_chunk = hidden_chunk @ all_item_emb.T

        # Restrict candidates to the same domain as each row's target item
        # (also implicitly excludes padding, since domain_id[0] == 0 and no real target has domain_id 0)
        target_domain_id_chunk = domain_id[targets_chunk]                            # (n,)
        wrong_domain_mask = domain_id.unsqueeze(0) != target_domain_id_chunk.unsqueeze(1)  # (n, num_items)
        logits_chunk = logits_chunk.masked_fill(wrong_domain_mask, float('-inf'))

        top_k_items = logits_chunk.topk(max_k, dim=1).indices

        for i in range(hidden_chunk.size(0)):
            recommended_items_by_user[global_idx] = top_k_items[i].tolist()
            relevant_items_by_user[global_idx] = [targets_chunk[i].item()]
            global_idx += 1

    return recommended_items_by_user, relevant_items_by_user

recommended_model, relevant_items = full_ranking_eval(model, all_hidden_test, all_targets_test, NUM_ITEMS,  chunk_size=128)

model_metrics = {
    "MAP@5": map_at_k(recommended_model, relevant_items, k=5),
    "NDCG@10": average_metric_over_users(ndcg_at_k, recommended_model, relevant_items, k=10),
    "Recall@10": average_metric_over_users(recall_at_k, recommended_model, relevant_items, k=10),
}
print("Model:", model_metrics)

Free GPU memory: 15.17 GB
Model: {'MAP@5': 0.0019152042185599867, 'NDCG@10': np.float64(0.0032822316191079893), 'Recall@10': 0.006620329601207699}


In [29]:
books_item_idx_np = books_item_idx.cpu().numpy()
movies_item_idx_np = movies_item_idx.cpu().numpy()

top_popular_books = train_df.loc[train_df['item_idx'].isin(books_item_idx_np), 'item_idx'].value_counts().index[:10].tolist()
top_popular_movies = train_df.loc[train_df['item_idx'].isin(movies_item_idx_np), 'item_idx'].value_counts().index[:10].tolist()

rng = np.random.default_rng(42)
recommended_popular = {}
recommended_random = {}
for i, target_item in enumerate(y_test):
    is_books_target = item_domain[target_item] == 'books'
    recommended_popular[i] = top_popular_books if is_books_target else top_popular_movies
    pool = books_item_idx_np if is_books_target else movies_item_idx_np
    recommended_random[i] = rng.choice(pool, size=10, replace=False).tolist()

# assumes model_metrics, popular_metrics, random_metrics dicts already computed, e.g.:
# model_metrics = {"MAP@5": ..., "NDCG@10": ..., "Recall@10": ...}
popular_metrics = {
    "MAP@5": map_at_k(recommended_popular, relevant_items, k=5),
    "NDCG@10": average_metric_over_users(ndcg_at_k, recommended_popular, relevant_items, k=10),
    "Recall@10": average_metric_over_users(recall_at_k, recommended_popular, relevant_items, k=10),
}
random_metrics = {
    "MAP@5": map_at_k(recommended_random, relevant_items, k=5),
    "NDCG@10": average_metric_over_users(ndcg_at_k, recommended_random, relevant_items, k=10),
    "Recall@10": average_metric_over_users(recall_at_k, recommended_random, relevant_items, k=10),
}

rows = []
for metric in model_metrics:
    model_val = model_metrics[metric]
    popular_val = popular_metrics[metric]
    random_val = random_metrics[metric]

    lift_popular = model_val / popular_val if popular_val > 0 else float('inf')
    lift_random = model_val / random_val if random_val > 0 else float('inf')

    rows.append({
        "Metric": metric,
        "Model": model_val,
        "Top-Popular": popular_val,
        "Random": random_val,
        "Lift vs Popular": lift_popular,
        "Lift vs Random": lift_random,
    })

df = pd.DataFrame(rows)

# nice formatting: small decimals for raw metrics, "x" suffix for lift, inf handled gracefully
formatted = df.copy()
for col in ["Model", "Top-Popular", "Random"]:
    formatted[col] = formatted[col].map(lambda v: f"{v:.6f}")
for col in ["Lift vs Popular", "Lift vs Random"]:
    formatted[col] = formatted[col].map(lambda v: "∞" if v == float('inf') else f"×{v:.2f}")

print(formatted.to_string(index=False))

   Metric    Model Top-Popular   Random Lift vs Popular Lift vs Random
    MAP@5 0.001915    0.001834 0.000037           ×1.04         ×51.46
  NDCG@10 0.003282    0.002998 0.000060           ×1.09         ×55.06
Recall@10 0.006620    0.006015 0.000118           ×1.10         ×56.13


In [30]:
def split_by_domain(relevant_items_by_user, item_domain):
    """Splits user indices into 'books' and 'movies' groups based on
    the domain of their relevant (held-out target) item."""
    books_users, movies_users = [], []
    for user_id, relevant in relevant_items_by_user.items():
        target_item = relevant[0]
        if item_domain[target_item] == 'books':
            books_users.append(user_id)
        else:
            movies_users.append(user_id)
    return books_users, movies_users


def subset_dict(d, user_ids):
    return {u: d[u] for u in user_ids}


def compute_metrics_for_subset(recommended, relevant, user_ids):
    rec_subset = subset_dict(recommended, user_ids)
    rel_subset = subset_dict(relevant, user_ids)
    return {
        "MAP@5": map_at_k(rec_subset, rel_subset, k=5),
        "NDCG@10": average_metric_over_users(ndcg_at_k, rec_subset, rel_subset, k=10),
        "Recall@10": average_metric_over_users(recall_at_k, rec_subset, rel_subset, k=10),
    }


books_users, movies_users = split_by_domain(relevant_items, item_domain)
print(f"Books users: {len(books_users)}, Movies users: {len(movies_users)}")

results_by_domain = {}
for domain_name, user_ids in (("Books", books_users), ("Movies", movies_users)):
    model_d = compute_metrics_for_subset(recommended_model, relevant_items, user_ids)
    popular_d = compute_metrics_for_subset(recommended_popular, relevant_items, user_ids)
    random_d = compute_metrics_for_subset(recommended_random, relevant_items, user_ids)

    rows = []
    for metric in model_d:
        model_val = model_d[metric]
        popular_val = popular_d[metric]
        random_val = random_d[metric]
        lift_popular = model_val / popular_val if popular_val > 0 else float('inf')
        lift_random = model_val / random_val if random_val > 0 else float('inf')
        rows.append({
            "Metric": metric,
            "Model": model_val,
            "Top-Popular": popular_val,
            "Random": random_val,
            "Lift vs Popular": lift_popular,
            "Lift vs Random": lift_random,
        })

    df_domain = pd.DataFrame(rows)
    formatted = df_domain.copy()
    for col in ["Model", "Top-Popular", "Random"]:
        formatted[col] = formatted[col].map(lambda v: f"{v:.6f}")
    for col in ["Lift vs Popular", "Lift vs Random"]:
        formatted[col] = formatted[col].map(lambda v: "∞" if v == float('inf') else f"×{v:.2f}")

    print(f"\n=== {domain_name} ===")
    print(formatted.to_string(index=False))
    results_by_domain[domain_name] = df_domain

Books users: 63034, Movies users: 64150

=== Books ===
   Metric    Model Top-Popular   Random Lift vs Popular Lift vs Random
    MAP@5 0.001645    0.001262 0.000045           ×1.30         ×36.16
  NDCG@10 0.002889    0.002172 0.000064           ×1.33         ×45.12
Recall@10 0.005997    0.004347 0.000111           ×1.38         ×54.00

=== Movies ===
   Metric    Model Top-Popular   Random Lift vs Popular Lift vs Random
    MAP@5 0.002181    0.002397 0.000029           ×0.91         ×74.96
  NDCG@10 0.003669    0.003810 0.000055           ×0.96         ×66.37
Recall@10 0.007233    0.007654 0.000125           ×0.95         ×58.00
